# Feature Analysis

## 1. Load Data

 ### 1.1 Load Data from Notebook 1

In [1]:
import pandas as pd
import numpy as np

team_seasons = pd.read_csv("../data/processed/team_seasons.csv")

team_seasons.shape

(2836, 24)

### 1.2 Load New Data

In [2]:
tourney_results = pd.read_csv("../data/raw/MNCAATourneyCompactResults.csv")

teams = pd.read_csv( "../data/raw/MTeams.csv")

 ## 2. Inspect Tournament Results

In [3]:
tourney_results.shape

(2585, 8)

In [4]:
tourney_results.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,136,1116,63,1234,54,N,0
1,1985,136,1120,59,1345,58,N,0
2,1985,136,1207,68,1250,43,N,0
3,1985,136,1229,58,1425,55,N,0
4,1985,136,1242,49,1325,38,N,0


In [5]:
tourney_results.columns

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT'],
      dtype='object')

In [6]:
teams.shape

(381, 4)

In [7]:
teams.head()

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2026
1,1102,Air Force,1985,2026
2,1103,Akron,1985,2026
3,1104,Alabama,1985,2026
4,1105,Alabama A&M,2000,2026


In [8]:
teams.columns

Index(['TeamID', 'TeamName', 'FirstD1Season', 'LastD1Season'], dtype='object')

## 3. Clean Tournament Results

### 3.1 Fix Years

In [9]:
years = [2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024]

tourney_results = tourney_results[tourney_results["Season"].isin(years)].copy()

In [10]:
tourney_results.shape

(535, 8)

In [11]:
tourney_results["Season"].value_counts().sort_index()

Season
2016    67
2017    67
2018    67
2019    67
2021    66
2022    67
2023    67
2024    67
Name: count, dtype: int64

### 3.2 Standardize Team Name Formatting

In [12]:
team_name_lookup = dict(zip(teams["TeamID"], teams["TeamName"]))

In [13]:
tourney_results["WTeam"] = tourney_results["WTeamID"].map(team_name_lookup)
tourney_results["LTeam"] = tourney_results["LTeamID"].map(team_name_lookup)

In [14]:
tourney_results[["Season", "WTeam", "WScore", "LTeam", "LScore"]].head(10)

,Season,WTeam,WScore,LTeam,LScore
1983,2016,FGCU,96,F Dickinson,65
1984,2016,Wichita St,70,Vanderbilt,50
1985,2016,Holy Cross,59,Southern Univ,55
1986,2016,Michigan,67,Tulsa,62
1987,2016,Ark Little Rock,85,Purdue,83
1988,2016,Butler,71,Texas Tech,61
1989,2016,Connecticut,74,Colorado,67
1990,2016,Duke,93,UNC Wilmington,85
1991,2016,Gonzaga,68,Seton Hall,52
1992,2016,Indiana,99,Chattanooga,74


In [15]:
tourney_results[["WTeam", "LTeam"]].isna().sum()

WTeam    0
LTeam    0
dtype: int64

### 3.3  Identify Unmatched Team Names

In [16]:
# Get every unique tournament team
tournament_teams = set(tourney_results["WTeam"]) | set(tourney_results["LTeam"])

season_teams = set(team_seasons["TEAM"])

In [17]:
# Find teams not in team_seasons dataset
unmatched_teams = sorted(tournament_teams - season_teams)

len(unmatched_teams)

66

In [18]:
unmatched_teams

['Abilene Chr',
 'Appalachian St',
 'Arizona St',
 'Ark Little Rock',
 'Boise St',
 'CS Bakersfield',
 'CS Fullerton',
 'Cleveland St',
 'Col Charleston',
 'Colorado St',
 'E Washington',
 'ETSU',
 'F Dickinson',
 'FGCU',
 'FL Atlantic',
 'Florida St',
 'Fresno St',
 'Georgia St',
 'Grambling',
 'Iowa St',
 'Jacksonville St',
 'Kansas St',
 'Kennesaw',
 'Kent',
 'Long Beach St',
 'Louisiana',
 'Loyola-Chicago',
 'MTSU',
 'McNeese St',
 'Michigan St',
 'Mississippi St',
 'Montana St',
 'Morehead St',
 "Mt St Mary's",
 'Murray St',
 'N Dakota St',
 'N Kentucky',
 'NC Central',
 'NC State',
 'New Mexico St',
 'Norfolk St',
 'Ohio St',
 'Oklahoma St',
 'Oregon St',
 'Penn St',
 'Prairie View',
 'S Dakota St',
 'SE Missouri St',
 'SF Austin',
 'San Diego St',
 'Southern Univ',
 'St Bonaventure',
 "St John's",
 "St Joseph's PA",
 'St Louis',
 "St Mary's CA",
 "St Peter's",
 'TAM C. Christi',
 'TX Southern',
 'Utah St',
 'WI Green Bay',
 'WKU',
 'Washington St',
 'Weber St',
 'Wichita St',
 '

### 3.4 Investigate Team Name Differences

In [19]:
# Create year-specific mapping
unmatched_pairs = []

for _, row in tourney_results.iterrows():
    for team in [row["WTeam"], row["LTeam"]]:
        year = row["Season"]

        match = team_seasons[
            (team_seasons["YEAR"] == year) &
            (team_seasons["TEAM"] == team)
        ]

        if match.empty:
            unmatched_pairs.append((year, team))

unmatched_pairs = sorted(set(unmatched_pairs))

len(unmatched_pairs)

154

In [20]:
unmatched_pairs

[(2016, 'Ark Little Rock'),
 (2016, 'CS Bakersfield'),
 (2016, 'F Dickinson'),
 (2016, 'FGCU'),
 (2016, 'Fresno St'),
 (2016, 'Iowa St'),
 (2016, 'MTSU'),
 (2016, 'Michigan St'),
 (2016, 'Oregon St'),
 (2016, 'S Dakota St'),
 (2016, 'SF Austin'),
 (2016, 'Southern Univ'),
 (2016, "St Joseph's PA"),
 (2016, 'WI Green Bay'),
 (2016, 'Weber St'),
 (2016, 'Wichita St'),
 (2017, 'ETSU'),
 (2017, 'FGCU'),
 (2017, 'Florida St'),
 (2017, 'Iowa St'),
 (2017, 'Jacksonville St'),
 (2017, 'Kansas St'),
 (2017, 'Kent'),
 (2017, 'MTSU'),
 (2017, 'Michigan St'),
 (2017, "Mt St Mary's"),
 (2017, 'N Kentucky'),
 (2017, 'NC Central'),
 (2017, 'New Mexico St'),
 (2017, 'Oklahoma St'),
 (2017, 'S Dakota St'),
 (2017, "St Mary's CA"),
 (2017, 'TX Southern'),
 (2017, 'Wichita St'),
 (2018, 'Arizona St'),
 (2018, 'CS Fullerton'),
 (2018, 'Col Charleston'),
 (2018, 'Florida St'),
 (2018, 'Georgia St'),
 (2018, 'Kansas St'),
 (2018, 'Loyola-Chicago'),
 (2018, 'Michigan St'),
 (2018, 'Murray St'),
 (2018, 'NC C

In [21]:
unmatched_names = sorted(set(team for year, team in unmatched_pairs))

len(unmatched_names), unmatched_names

(66,
 ['Abilene Chr',
  'Appalachian St',
  'Arizona St',
  'Ark Little Rock',
  'Boise St',
  'CS Bakersfield',
  'CS Fullerton',
  'Cleveland St',
  'Col Charleston',
  'Colorado St',
  'E Washington',
  'ETSU',
  'F Dickinson',
  'FGCU',
  'FL Atlantic',
  'Florida St',
  'Fresno St',
  'Georgia St',
  'Grambling',
  'Iowa St',
  'Jacksonville St',
  'Kansas St',
  'Kennesaw',
  'Kent',
  'Long Beach St',
  'Louisiana',
  'Loyola-Chicago',
  'MTSU',
  'McNeese St',
  'Michigan St',
  'Mississippi St',
  'Montana St',
  'Morehead St',
  "Mt St Mary's",
  'Murray St',
  'N Dakota St',
  'N Kentucky',
  'NC Central',
  'NC State',
  'New Mexico St',
  'Norfolk St',
  'Ohio St',
  'Oklahoma St',
  'Oregon St',
  'Penn St',
  'Prairie View',
  'S Dakota St',
  'SE Missouri St',
  'SF Austin',
  'San Diego St',
  'Southern Univ',
  'St Bonaventure',
  "St John's",
  "St Joseph's PA",
  'St Louis',
  "St Mary's CA",
  "St Peter's",
  'TAM C. Christi',
  'TX Southern',
  'Utah St',
  'WI Gr

In [22]:
from difflib import get_close_matches

for team in unmatched_names:
    years_for_team = [
        year for year, name in unmatched_pairs
        if name == team
    ]

    possible_names = team_seasons[team_seasons["YEAR"].isin(years_for_team)]["TEAM"].unique()

    matches = get_close_matches(team, possible_names, n=3,cutoff=0.5)

    print(team, "->", matches)

Abilene Chr -> ['Abilene Christian']
Appalachian St -> ['Appalachian St.', 'Michigan St.', 'Coppin St.']
Arizona St -> ['Arizona St.', 'Arizona', 'Oregon St.']
Ark Little Rock -> ['Arkansas Little Rock']
Boise St -> ['Boise St.', 'San Jose St.', 'Illinois St.']
CS Bakersfield -> ['Cal St. Bakersfield', 'Fairfield', 'UC Riverside']
CS Fullerton -> ['Cal St. Fullerton', 'Clemson', 'Southern']
Cleveland St -> ['Cleveland St.', 'Portland St.', 'Morehead St.']
Col Charleston -> ['College of Charleston', 'Charleston Southern', 'Coastal Carolina']
Colorado St -> ['Colorado St.', 'Colorado', 'Florida St.']
E Washington -> ['Washington', 'Eastern Washington', 'Washington St.']
ETSU -> ['TCU', 'SMU', 'LSU']
F Dickinson -> ['Fairleigh Dickinson', 'Davidson', 'Old Dominion']
FGCU -> ['VCU', 'TCU', 'FIU']
FL Atlantic -> ['Florida Atlantic']
Florida St -> ['Florida St.', 'Florida', 'Florida A&M']
Fresno St -> ['Fresno St.', 'Oregon St.', 'Sacramento St.']
Georgia St -> ['Georgia St.', 'Georgia', 'Ge

In [23]:
for team in unmatched_names:
    years_for_team = [
        year for year, name in unmatched_pairs
        if name == team
    ]

    print("\n", team, years_for_team)

    for year in years_for_team:
        names = team_seasons[
            team_seasons["YEAR"] == year
        ]["TEAM"].tolist()

        matches = get_close_matches(
            team,
            names,
            n=5,
            cutoff=0.3
        )

        print(year, matches)


 Abilene Chr [2019, 2021]
2019 ['Abilene Christian', 'Denver', 'College of Charleston', 'Winthrop', 'Tennessee Tech']
2021 ['Abilene Christian', 'Denver', 'College of Charleston', 'Winthrop', 'Tennessee Tech']

 Appalachian St [2021]
2021 ['Appalachian St.', 'Michigan St.', 'Coppin St.', 'Alcorn St.', 'Wichita St.']

 Arizona St [2018, 2019, 2023]
2018 ['Arizona St.', 'Arizona', 'Oregon St.', 'Alcorn St.', 'Montana St.']
2019 ['Arizona St.', 'Arizona', 'Oregon St.', 'Alcorn St.', 'Montana St.']
2023 ['Arizona St.', 'Arizona', 'Oregon St.', 'Alcorn St.', 'Montana St.']

 Ark Little Rock [2016]
2016 ['Arkansas Little Rock', 'Air Force', 'Arkansas Pine Bluff', 'Oral Roberts', 'Arkansas St.']

 Boise St [2022, 2023, 2024]
2022 ['Boise St.', 'San Jose St.', 'Illinois St.', 'Coppin St.', 'McNeese St.']
2023 ['Boise St.', 'San Jose St.', 'Illinois St.', 'Coppin St.', 'McNeese St.']
2024 ['Boise St.', 'San Jose St.', 'Illinois St.', 'Coppin St.', 'McNeese St.']

 CS Bakersfield [2016]
2016 ['

In [24]:
search_terms = [
    "Tennessee", "Florida Gulf", "Middle Tennessee",
    "Western Kentucky", "Carolina", "Louisiana",
    "Southern", "Green Bay"
]

for term in search_terms:
    print(
        term,
        sorted(
            team_seasons[
                team_seasons["TEAM"].str.contains(term, case=False, na=False)
            ]["TEAM"].unique()
        )
    )

Tennessee ['East Tennessee St.', 'Middle Tennessee', 'Tennessee', 'Tennessee Martin', 'Tennessee St.', 'Tennessee Tech']
Florida Gulf ['Florida Gulf Coast']
Middle Tennessee ['Middle Tennessee']
Western Kentucky ['Western Kentucky']
Carolina ['Coastal Carolina', 'East Carolina', 'North Carolina', 'North Carolina A&T', 'North Carolina Central', 'North Carolina St.', 'South Carolina', 'South Carolina St.', 'Western Carolina']
Louisiana ['Louisiana Lafayette', 'Louisiana Monroe', 'Louisiana Tech', 'Southeastern Louisiana']
Southern ['Charleston Southern', 'Georgia Southern', 'Southern', 'Southern Illinois', 'Southern Indiana', 'Southern Miss', 'Southern Utah', 'Texas Southern']
Green Bay ['Green Bay']


### 3.5 Create Team Name Mapping

In [25]:
tournament_name_map = {
    "ETSU": "East Tennessee St.",
    "FGCU": "Florida Gulf Coast",
    "MTSU": "Middle Tennessee",
    "WKU": "Western Kentucky",
    "NC State": "North Carolina St.",
    "WI Green Bay": "Green Bay",

    "Abilene Chr": "Abilene Christian",
    "Appalachian St": "Appalachian St.",
    "Arizona St": "Arizona St.",
    "Ark Little Rock": "Arkansas Little Rock",
    "Boise St": "Boise St.",
    "CS Bakersfield": "Cal St. Bakersfield",
    "CS Fullerton": "Cal St. Fullerton",
    "Cleveland St": "Cleveland St.",
    "Col Charleston": "College of Charleston",
    "Colorado St": "Colorado St.",
    "E Washington": "Eastern Washington",
    "F Dickinson": "Fairleigh Dickinson",
    "FL Atlantic": "Florida Atlantic",
    "Florida St": "Florida St.",
    "Fresno St": "Fresno St.",
    "Georgia St": "Georgia St.",
    "Grambling": "Grambling St.",
    "Iowa St": "Iowa St.",
    "Jacksonville St": "Jacksonville St.",
    "Kansas St": "Kansas St.",
    "Kennesaw": "Kennesaw St.",
    "Kent": "Kent St.",
    "Long Beach St": "Long Beach St.",
    "Louisiana": "Louisiana Lafayette",
    "Loyola-Chicago": "Loyola Chicago",
    "McNeese St": "McNeese St.",
    "Michigan St": "Michigan St.",
    "Mississippi St": "Mississippi St.",
    "Montana St": "Montana St.",
    "Morehead St": "Morehead St.",
    "Mt St Mary's": "Mount St. Mary's",
    "Murray St": "Murray St.",
    "N Dakota St": "North Dakota St.",
    "N Kentucky": "Northern Kentucky",
    "NC Central": "North Carolina Central",
    "New Mexico St": "New Mexico St.",
    "Norfolk St": "Norfolk St.",
    "Ohio St": "Ohio St.",
    "Oklahoma St": "Oklahoma St.",
    "Oregon St": "Oregon St.",
    "Penn St": "Penn St.",
    "Prairie View": "Prairie View A&M",
    "S Dakota St": "South Dakota St.",
    "SE Missouri St": "Southeast Missouri St.",
    "SF Austin": "Stephen F. Austin",
    "San Diego St": "San Diego St.",
    "Southern Univ": "Southern",
    "St Bonaventure": "St. Bonaventure",
    "St John's": "St. John's",
    "St Joseph's PA": "Saint Joseph's",
    "St Louis": "Saint Louis",
    "St Mary's CA": "Saint Mary's",
    "St Peter's": "Saint Peter's",
    "TAM C. Christi": "Texas A&M Corpus Chris",
    "TX Southern": "Texas Southern",
    "Utah St": "Utah St.",
    "Washington St": "Washington St.",
    "Weber St": "Weber St.",
    "Wichita St": "Wichita St.",
    "Wright St": "Wright St."
}

In [26]:
tourney_results["WTeam"] = tourney_results["WTeam"].replace(tournament_name_map)
tourney_results["LTeam"] = tourney_results["LTeam"].replace(tournament_name_map)

### 3.6 Apply Team Name Mapping

In [27]:
unmatched_pairs = []

for _, row in tourney_results.iterrows():
    for team in [row["WTeam"], row["LTeam"]]:
        year = row["Season"]

        match = team_seasons[
            (team_seasons["YEAR"] == year) &
            (team_seasons["TEAM"] == team)
        ]

        if match.empty:
            unmatched_pairs.append((year, team))

unmatched_pairs = sorted(set(unmatched_pairs))

len(unmatched_pairs), unmatched_pairs

(0, [])

In [28]:
tourney_results.columns

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WTeam', 'LTeam'],
      dtype='object')

## 4. Match Tournament Teams to Team-Season Data

In [29]:
games = tourney_results[[
    "Season",
    "DayNum",
    "WTeam",
    "LTeam",
    "WScore",
    "LScore"
]].copy()

games = games.rename(columns={
    "Season": "YEAR"
})

games.head()

,YEAR,DayNum,WTeam,LTeam,WScore,LScore
1983,2016,134,Florida Gulf Coast,Fairleigh Dickinson,96,65
1984,2016,134,Wichita St.,Vanderbilt,70,50
1985,2016,135,Holy Cross,Southern,59,55
1986,2016,135,Michigan,Tulsa,67,62
1987,2016,136,Arkansas Little Rock,Purdue,85,83


In [30]:
print(games.shape)
print(games.isna().sum())

(535, 6)
YEAR      0
DayNum    0
WTeam     0
LTeam     0
WScore    0
LScore    0
dtype: int64


### 4.1 Attach Winning Team Statistics

In [31]:
winner_stats = team_seasons.add_prefix("W_")

games = games.merge(
    winner_stats,
    left_on=["WTeam", "YEAR"],
    right_on=["W_TEAM", "W_YEAR"],
    how="left",
    validate="many_to_one"
)

In [32]:
games.shape

(535, 30)

In [33]:
games[["WTeam", "YEAR", "W_TEAM", "W_YEAR"]].head()

,WTeam,YEAR,W_TEAM,W_YEAR
0,Florida Gulf Coast,2016,Florida Gulf Coast,2016
1,Wichita St.,2016,Wichita St.,2016
2,Holy Cross,2016,Holy Cross,2016
3,Michigan,2016,Michigan,2016
4,Arkansas Little Rock,2016,Arkansas Little Rock,2016


In [34]:
games["W_TEAM"].isna().sum()

np.int64(0)

 ### 4.2 Attach Losing Team Statistics

In [35]:
loser_stats = team_seasons.add_prefix("L_")

games = games.merge(
    loser_stats,
    left_on=["LTeam", "YEAR"],
    right_on=["L_TEAM", "L_YEAR"],
    how="left",
    validate="many_to_one"
)

In [36]:
games["L_TEAM"].isna().sum()

np.int64(0)

In [37]:
games[[
    "YEAR",
    "WTeam",
    "LTeam",
    "W_SEED",
    "L_SEED",
    "W_POWER_RATING",
    "L_POWER_RATING"
]].head()

,YEAR,WTeam,LTeam,W_SEED,L_SEED,W_POWER_RATING,L_POWER_RATING
0,2016,Florida Gulf Coast,Fairleigh Dickinson,16.0,16.0,71.4,66.7
1,2016,Wichita St.,Vanderbilt,11.0,11.0,86.6,85.6
2,2016,Holy Cross,Southern,16.0,16.0,66.8,68.0
3,2016,Michigan,Tulsa,11.0,11.0,79.6,79.9
4,2016,Arkansas Little Rock,Purdue,12.0,5.0,78.9,88.7


## 5. Construct Matchup Dataset

### 5.1 Assign Teams to A and B

In [38]:
# Randomly assign the historical winner to A or B using a fixed seed for reproducibilityrng = np.random.default_rng(42)

winner_is_a = rng.integers(
    0,
    2,
    size=len(games)
).astype(bool)

NameError: name 'rng' is not defined

In [ ]:
#creating A and B targets
games["TEAM_A"] = np.where(
    winner_is_a,
    games["WTeam"],
    games["LTeam"]
)

games["TEAM_B"] = np.where(
    winner_is_a,
    games["LTeam"],
    games["WTeam"]
)

games["TEAM_A_WIN"] = winner_is_a.astype(int)

In [ ]:
#checking A and B targets
games[[
    "YEAR",
    "WTeam",
    "LTeam",
    "TEAM_A",
    "TEAM_B",
    "TEAM_A_WIN"
]].head(10)

In [ ]:
games["TEAM_A_WIN"].value_counts()

### 5.2 Assign Team Statistics to A and B

In [ ]:
#adding seed
games["A_SEED"] = np.where(
    winner_is_a,
    games["W_SEED"],
    games["L_SEED"]
)

games["B_SEED"] = np.where(
    winner_is_a,
    games["L_SEED"],
    games["W_SEED"]
)

In [ ]:
# adding power rating
games["A_POWER_RATING"] = np.where(
    winner_is_a,
    games["W_POWER_RATING"],
    games["L_POWER_RATING"]
)

games["B_POWER_RATING"] = np.where(
    winner_is_a,
    games["L_POWER_RATING"],
    games["W_POWER_RATING"]
)

In [ ]:
#checking
games[[
    "TEAM_A",
    "TEAM_B",
    "A_SEED",
    "B_SEED",
    "A_POWER_RATING",
    "B_POWER_RATING",
    "TEAM_A_WIN"
]].head(10)

In [ ]:
# adding other stats
stat_cols = [
    col for col in team_seasons.columns
    if col not in ["TEAM", "YEAR"]
]

for col in stat_cols:
    games[f"A_{col}"] = np.where(
        winner_is_a,
        games[f"W_{col}"],
        games[f"L_{col}"]
    )

    games[f"B_{col}"] = np.where(
        winner_is_a,
        games[f"L_{col}"],
        games[f"W_{col}"]
    )

In [ ]:
games[[
    "TEAM_A",
    "TEAM_B",
    "A_SEED",
    "B_SEED",
    "A_KADJ_O",
    "B_KADJ_O",
    "A_POWER_RATING",
    "B_POWER_RATING",
    "TEAM_A_WIN"
]].head()

In [ ]:
print(games.shape)
print(games.filter(regex="^A_").columns.tolist())
print(games.filter(regex="^B_").columns.tolist())

## 6. Create Difference Features

### 6.1 Create Win Percentage

In [ ]:
games["A_WIN_PCT"] = games["A_W"] / games["A_G"]
games["B_WIN_PCT"] = games["B_W"] / games["B_G"]

In [ ]:
games[[
    "TEAM_A",
    "A_W",
    "A_G",
    "A_WIN_PCT",
    "TEAM_B",
    "B_W",
    "B_G",
    "B_WIN_PCT"
]].head()

### 6.2 Create Statistical Differences

In [ ]:
# Create differences for useful statistics
diff_stats = [
    "SEED",
    "WIN_PCT",
    "2P_O",
    "2P_D",
    "3P_O",
    "3P_D",
    "ADJ_T",
    "WAB",
    "KADJ_O",
    "KADJ_D",
    "BARTHAG",
    "EFG%",
    "EFG%D",
    "TOV%",
    "TOV%D",
    "OREB%",
    "DREB%",
    "FTR",
    "FTRD",
    "POWER_RATING",
    "AP_RANK"
]
for stat in diff_stats:
    games[f"{stat}_DIFF"] = games[f"A_{stat}"] - games[f"B_{stat}"]

### 6.3 Validate Difference Features

In [ ]:
games[[
    "TEAM_A",
    "TEAM_B",
    "SEED_DIFF",
    "WIN_PCT_DIFF",
    "KADJ_O_DIFF",
    "KADJ_D_DIFF",
    "BARTHAG_DIFF",
    "POWER_RATING_DIFF",
    "TEAM_A_WIN"
]].head(10)

In [ ]:
diff_columns = [f"{stat}_DIFF" for stat in diff_stats]

print(len(diff_columns))
print(diff_columns)

## 7. Check Missing Values

### 7.1 Check Missing Values in Difference Features

In [ ]:
missing_diff = games[diff_columns].isna().sum()

missing_diff[missing_diff > 0]

In [ ]:
missing_pct = (
    games[diff_columns].isna().mean() * 100
)

missing_pct[missing_pct > 0].sort_values(ascending=False)

### 7.2 Select Complete Model Features

#### AP_RANK_DIFF is excluded because approximately 25% of tournament matchups are missing AP ranking data. WIN_PCT_DIFF is excluded because the CBB season totals include postseason games, which could introduce data leakage when predicting earlier tournament games. A pre-tournament win percentage will be investigated in a later analysis.

In [ ]:
model_features = [
    col for col in diff_columns
    if col not in ["AP_RANK_DIFF", "WIN_PCT_DIFF"]
]
print(len(model_features))
print(games[model_features].isna().sum().sum())

## Section 9. Explore Matchup Features

### 9.1 Summary Statistics

In [ ]:
games[model_features].describe().T

### 9.2 Correlation with Winning

In [ ]:
feature_correlations = (
    games[model_features + ["TEAM_A_WIN"]]
    .corr()["TEAM_A_WIN"]
    .drop("TEAM_A_WIN")
    .sort_values()
)

feature_correlations

 ### 9.3 Visualize Strong Features

In [ ]:
import matplotlib.pyplot as plt

feature_correlations.sort_values().plot(
    kind="barh",
    figsize=(8, 7)
)

plt.xlabel("Correlation with Team A Win")
plt.ylabel("Feature")
plt.title("Feature Correlation with Tournament Game Outcome")
plt.tight_layout()
plt.show()

## 10. Review Target Variable

### 10.1 Target Distribution

In [ ]:
target_counts = games["TEAM_A_WIN"].value_counts().sort_index()
target_pct = games["TEAM_A_WIN"].value_counts(normalize=True).sort_index() * 100

print("Counts:")
print(target_counts)

print("\nPercentages:")
print(target_pct)

## 11 Final data check

### 11.1 Check Dataset Size

In [ ]:
print("Games:", len(games))
print("Model features:", len(model_features))

### 11.2 Check Missing Model Data

In [ ]:
print("Missing model values:")
print(games[model_features].isna().sum().sum())

### 11.3 Check Target

In [ ]:
print("Missing targets:", games["TEAM_A_WIN"].isna().sum())
print("Target values:", sorted(games["TEAM_A_WIN"].unique()))

### 11.4 Check Duplicate Games

In [ ]:
games.duplicated(
    subset=["YEAR", "DayNum", "TEAM_A", "TEAM_B"]
).sum()

### 11.5 Check Seasons

In [ ]:
sorted(games["YEAR"].unique())

## 12. Export Matchups

### 12.1 Create Final Dataset

In [ ]:
final_columns = [
    "YEAR",
    "DayNum",
    "TEAM_A",
    "TEAM_B",
    "TEAM_A_WIN"
] + model_features

tournament_matchups = games[final_columns].copy()

tournament_matchups.head()

### 12.2 final check

In [ ]:
print(tournament_matchups.shape)
print(tournament_matchups.isna().sum().sum())

### 12.3 Export

In [ ]:
tournament_matchups.to_csv(
    "../data/final/tournament_matchups.csv",
    index=False
)